## 생성형 AI

#### AI는 **새로운 이미지** 를 만들어낼 수 있는가?

- 기존 AI : Discriminative(판별자). 이미지를 보고 판단하는 것에 집중.
- 생성형 AI : Generative(생성자). 새로운 이미지를 창조. 

### Latent Space(가능성 공간)

데이터의 핵심 특징을 압축해서 표현한 숨겨진 공간

가령 어떤 28 * 28 입력 이미지를 신경망을 통과시켜 z = [0.2, -1.4, 0.8, 2.1] 이런 형식의 최종 출력을 얻었다고 하면

이 벡터가 Latent Vector가 되고, 이런 벡터들로 구성된 공간을 Latent Space 라고 한다.

### 1. GAN(Generated Adversarial Network)


- 생성자(Generator) : 가짜 이미지 생성
- 판별자(Discriminator) : 진짜/가짜 판별

![GAN_vs_Discriminator](imgs/CV0207/GAN_vs_Discriminator.png)

$$\min_{G} \max_{D} V(D, G) = \mathbb{E}_{x \sim p_{data}(x)} [\log D(x)] + \mathbb{E}_{z \sim p_{z}(z)} [\log(1 - D(G(z)))]$$

| 항목      | 의미        |
| ------- | --------- |
| D(x)    | 진짜 이미지를 True로 판별할 확률|
| D(G(z)) | 가짜 이미지를 True로 판별할 확률 |


$\mathbb{E}$ : 평균. 배치 이미지 여러 개의 Loss 평균으로 생각하면 된다.

D는 V를 최대화하려 하고, G는 V를 최소화하려 한다.

### GAN vs DCGAN

GAN : MLP. 즉 입력 이미지를 1차원으로 펼쳐서 입력.

DCGAN : 입력 이미지를 CNN처럼 2차원으로 그대로 입력.

### GAN의 문제점

1. 학습 불안정 : G가 좋아지면 D가 틀리고, D가 좋아지면 G가 틀리는 구조.
2. 모드 붕괴(Mode Collapse) : G는 특정 이미지가 D를 잘 속이는 것을 발견하면 해당 이미지들만 반복 생성한다. 즉 다양성이 사라진다. 

### GAN의 발전사

GAN
→ DCGAN
→ Conditional GAN
→ Pix2Pix / CycleGAN
→ WGAN
→ StyleGAN
→ Diffusion

### VAE(Variational AutoEncoder)

이미지를 잠재공간의 확률분포로 변환하고, 그 분포에서 샘플링해 이미지를 복원/생성하는 모델

![VAE](imgs/CV0207/VAE1.png)

VAE가 있기 전 AE(AutoEncoder)가 있었음.

![AE](imgs/CV0207/AE1.png)

AE의 구조를 예시로 MNIST 이미지 입력으로 보여주면

784차원 이미지 → Encoder → 20차원 latent vector z → Decoder → 784차원 복원 이미지

정도가 된다.

Encoder : 784 → 400 → 20 처럼 더 작은 벡터로 압축하는 신경망

Decoder : 20 → 400 → 784 처럼 latent vector z를 다시 원래 크기로 복원시키는 신경망

AE의 목표는 입력 x와 출력 x'가 최대한 비슷해지게 만드는 것이다.


AE는 Encoder에서 z를 생성했다면

VAE는 Encoder에서 z 대신 $\mu$, $\sigma$ 를 생성한다.

그리고 이 $\mu$, $\sigma$ 를 이용해 z를 생성하는데,

이 과정을 **재매개변수화(reparameterization)** 이라고 한다.

$z = \mu + \sigma \cdot \epsilon  (\epsilon \sim N(0, 1))$

가령 예를 들어서 Encoder : 784 → 400 → 20 구조를 예시로 들어보면

일단 `self.fc = nn.Linear(784, 400)`로 신경망을 한번 통과시키고

이후 `self.mu = nn.Linear(400, 20)`, `self.var = nn.Linear(400, 20)`로 

평균과 분산(정확히는 분산의 로그값)으로 쓸 값들을 생성한다.

이 때, 분산은 $\sigma^2$ 으로 항상 양수가 나와야 하나 출력층의 값들은 음수가 나올 수도 있기 때문에

log(σ²)를 통해 스케일차이를 안정화 시켜 준 다음(값이 너무 커지거나 작은 경우를 대비한 분포를 줄여주는 스킬 정도로 생각하면 됨)

$\sigma = e ^ {\frac{1}{2} \log {\sigma ^ 2}}$ 공식을 적용. (코드는 `std = torch.exp(0.5 * var)`)

$z = \mu + \sigma \cdot \epsilon  (\epsilon \sim N(0, 1))$

에서 사용할 랜덤 노이즈 $\epsilon$ 을 생성. (코드는 `eps = torch.randn_like(std)`)

이런 식으로 잠재벡터(Latent Vector) z를 생성한다.